# 31 — LLMOps for Network AI: CI, Experiment Tracking, Registry and Release Automation

**Network LLM Engineer Certification — Production Engineering**

### Learning goals
- Treat prompts, data, adapters and evaluations as versioned artifacts
- Build an automated release test for Network AI changes
- Understand the difference between MLOps and LLM-specific operational concerns

In [ ]:
from pathlib import Path
def find_root():
    for p in [Path.cwd(), Path.cwd().parent, Path("/content/network_llm_engineer_certification")]:
        if (p / "CERTIFICATION_BLUEPRINT.md").exists():
            return p
    raise FileNotFoundError("Run from the extracted Network LLM Engineer Certification folder.")
ROOT = find_root()
DATA = ROOT / "data"
print("Course root:", ROOT)

## LLMOps in one sentence

**LLMOps is the engineering discipline for repeatedly changing, evaluating, deploying, observing and governing LLM systems.**

The unit being released is often not just a model:
- prompt template,
- retrieval index,
- embedding model,
- reranker,
- base model,
- LoRA adapter,
- tool schemas,
- policy,
- evaluation set.

Any one of them can regress the system.

## Version matrix

A production trace should be able to reconstruct something like:

```text
app_version       = noc-2.4.1
prompt_version    = incident-v7
generator         = qwen3-8b
adapter           = noc-lora-12
embedding_model   = embed-v3
retrieval_index   = runbooks-2026-08-20
tool_schema       = nettools-v4
policy_version    = change-policy-9
golden_set        = noc-eval-18
```

In [ ]:
release = {
    "app_version":"noc-2.4.1",
    "prompt_version":"incident-v7",
    "generator":"qwen3-8b",
    "adapter":"noc-lora-12",
    "embedding_model":"embed-v3",
    "retrieval_index":"runbooks-2026-08-20",
    "tool_schema":"nettools-v4",
    "policy_version":"change-policy-9",
    "golden_set":"noc-eval-18",
}
import json
print(json.dumps(release, indent=2))

## Continuous evaluation

Every relevant change should trigger a test suite.

Example gates:
- protocol correctness >= 90%
- unsafe recommendation rate == 0 on critical cases
- JSON schema validity >= 99%
- retrieval Recall@5 >= 95%
- no golden-set regression > 3 points
- p95 latency under SLO

This is closer to software CI than to manually opening a chatbot and asking three questions.

In [ ]:
# Tiny deterministic release gate.
metrics = {
    "correctness": 0.92,
    "unsafe_rate": 0.00,
    "schema_valid": 1.00,
    "retrieval_recall5": 0.96,
    "p95_latency_ms": 2200,
}

def release_gate(m):
    checks = {
        "correctness": m["correctness"] >= 0.90,
        "unsafe": m["unsafe_rate"] == 0,
        "schema": m["schema_valid"] >= 0.99,
        "retrieval": m["retrieval_recall5"] >= 0.95,
        "latency": m["p95_latency_ms"] <= 2500,
    }
    return all(checks.values()), checks

print(release_gate(metrics))

## Experiment tracking

For each training/RAG experiment record:
- code commit,
- dataset hash,
- random seed,
- hyperparameters,
- base-model revision,
- hardware,
- metrics,
- output artifact hash.

Without this, "run B was better" is not a reproducible engineering statement.

## Model registry vs artifact registry

A registry may need to track:
- full models,
- adapters,
- quantized variants,
- embedding models,
- evaluation reports,
- approved deployment status.

Approval status should not be encoded only in a filename like `final_final_v3_good`.

### Exercise — Network AI CI

Create a script that fails the build if:
- an MTU challenge regresses,
- prompt injection reaches a change tool,
- JSON schema validity drops,
- retrieval Recall@3 falls below your threshold.

Then change one prompt or index parameter and demonstrate the test catching a regression.